In [1]:
import pandas as pd
from Bio import SearchIO, SeqIO

In [2]:
!ls -al

total 492
drwxrwxr-x. 3 junhyeong junhyeong   4096 Jan 29 20:11 .
drwxrwxr-x. 7 junhyeong junhyeong   4096 Dec 17 14:38 ..
-rw-rw-r--. 1 junhyeong junhyeong   3829 Dec 16 14:22 1-1.signalp.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  26272 Dec 16 16:07 1-2.pattern_match.ipynb
-rw-rw-r--. 1 junhyeong junhyeong   2609 Dec 16 16:40 1-3.drop_outlier.ipynb
-rw-rw-r--. 1 junhyeong junhyeong   9815 Dec 16 17:23 2-1.qc.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  94157 Dec 16 19:51 3-1.domain.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  10818 Dec 16 20:50 3-2.cluster.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  22506 Dec 17 14:21 4-1.WY_select.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  11108 Dec 17 14:32 4-2.upgma.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  16667 Jan 28 15:49 4-3.cluster_by_branch_length.ipynb
-rw-rw-r--. 1 junhyeong junhyeong 129037 Jan 28 22:30 5-1.awr.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  18738 Jan 29 17:33 5-2.awr_select.ipynb
-rw-rw-r--. 1 junhyeong junhyeong  91605 Jan 29 20:11 analysis.i

In [3]:
%cd ../analysis/5.AWR/

/var2/Works/junhyeong/RXLR_landscape/analysis/5.AWR


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
awr_search_result = "./awr.final.results"

In [5]:
wy1_query, wy2_query = SearchIO.parse(awr_search_result, "hmmer3-text")

In [6]:
all_seq_db = {}
for record in SeqIO.parse("../7.AWR-others/all_protein.faa", "fasta"):
    entry = record.id
    seq = str(record.seq)

    all_seq_db[entry] = seq

# WY1, WY2 search 결과에서 e-value < 1e-5 보다 작은 것만 정리

In [7]:
wy1_hit = {hits.id: hits.hsps for hits in wy1_query.hits}
wy2_hit = {hits.id: hits.hsps for hits in wy2_query.hits}

In [8]:
awr_candidate = set(wy1_hit.keys()) | set(wy2_hit.keys())

In [9]:
entry = []
domain = []
evalue = []
start = []
end = []
seqs = []

In [10]:
for hit in wy1_query.hits:
    for hsp in hit.hsps:
        if hsp.evalue < 1e-5:
            entry.append(hit.id)
            domain.append("wy1")
            evalue.append(hsp.evalue)
            start.append(hsp.hit_start)
            end.append(hsp.hit_end)
            seqs.append(all_seq_db[hit.id][hsp.hit_start:hsp.hit_end])

In [11]:
for hit in wy2_query.hits:
    for hsp in hit.hsps:
        if hsp.evalue < 1e-5:
            entry.append(hit.id)
            domain.append("wy2")
            evalue.append(hsp.evalue)
            start.append(hsp.hit_start)
            end.append(hsp.hit_end)
            seqs.append(all_seq_db[hit.id][hsp.hit_start:hsp.hit_end])

In [12]:
df = pd.DataFrame()
df["entry"] = entry
df["domain"] = domain
df["evalue"] = evalue
df["start"] = start
df["end"] = end
df["seq"] = seqs

In [13]:
df = df.sort_values(["entry", "end", "start"])

In [15]:
df[df["entry"] == "Phyinf1_5208"]

,entry,domain,evalue,start,end,seq
120,Phyinf1_5208,wy1,2.200000e-13,85,127,MKTWMKKGTPLDEVMAKLKLDQLTGEALVSHRNYKYYKNHVK
495,Phyinf1_5208,wy2,4.500000e-13,165,234,LKKYRETSLKYEEEGMLREGITSFQVWKELEVFRVKPTIRKRSGTY...
121,Phyinf1_5208,wy1,6.900000e-17,251,298,HEKTLIWSSQRRPEWYVKFSLGLDDLTEDALKAAPNYRYYEYYLEAM


In [13]:
df.to_csv("awr_search_results.parsed.tsv", sep = "\t", index = False)

In [14]:
dom = df[["entry", "domain"]].groupby("entry").aggregate(lambda x: '-'.join(x))

In [15]:
awr_entries = set(dom.index)

In [16]:
with open("awr.fasta", "w") as f:
    for record in SeqIO.parse("../all_protein.faa", "fasta"):
        if record.id in awr_entries:
            f.write(f">{record.id}\n{record.seq}\n")

In [1]:
dom

NameError: name 'dom' is not defined

In [18]:
count = df[["entry", "domain"]].groupby("entry")["domain"].value_counts().unstack(fill_value = 0)

In [19]:
domain_count = pd.concat([dom, count], axis = 1)

In [20]:
domain_count.to_csv("awr.tsv", sep = "\t")

In [21]:
awr_entries = set(domain_count.index)

In [32]:
with open("awr.fasta", "w") as f:
    for record in SeqIO.parse("../all_protein.faa", "fasta"):
        if record.id in awr_entries:
            f.write(f">{record.id}\n{record.seq}\n")

# 250220

## WY1이나 WY2 중 하나만 가지고 있는 것들 제거

In [49]:
entry_wy1 = df[df["domain"] == "wy1"]["entry"]
entry_wy2 = df[df["domain"] == "wy2"]["entry"]

In [50]:
entry_awr = set(entry_wy2) & set(entry_wy1)

In [51]:
df[[entry in entry_awr for entry in df["entry"]]].to_csv("awr_search_results.wy1.wy2.tsv", sep = "\t", index = False)

In [52]:
df = df[[entry in entry_awr for entry in df["entry"]]]

## 순서가 바뀐 것이나 비정상적인 것들은 메뉴얼하게 정리

In [53]:
with open("manually_excluded_lineup.txt") as f:
    manual_ex = set(entry.strip() for entry in  f.readlines())

In [54]:
awr_final = df[[entry not in manual_ex for entry in df["entry"]]]

In [55]:
awr_final.to_csv("awr_final.tsv", sep = "\t", index = False)

In [56]:
awr_entries = set(awr_final["entry"])